# **Day 1: From Embeddings to Transformers**

**LLMs for Social Science** · Oxford Spring School

---

## The Course at a Glance

This is a five-day course that takes you from the foundations of how language models work to building research pipelines with them. Here is the full arc:

| Day | Theme | What You Learn |
|-----|-------|----------------|
| **→ Day 1** | **Foundations** | How models represent meaning, process text, and generate language |
| Day 2 | From Models to Tools | Post-training (RLHF, DPO), prompting, reasoning, model evaluation |
| Day 3 | Deploying for Research | Fine-tuning, APIs and batching, text classification and validation |
| Day 4 | Social Science Applications | Information extraction, RAG, LLMs as simulated agents |
| Day 5 | Agentic Workflows | Tool use, autonomous research assistants, capstone project |

Each day builds on the previous one. Today you learn **how language models work under the hood**. This matters because:

- When you **write prompts** (Day 2), you are writing for a next-token predictor: understanding that shapes how you frame instructions.
- When you **choose a model** (Day 2–3), you are navigating tradeoffs in scale, cost, and capability that trace back to the architecture.
- When you **build classification pipelines** (Day 3), you will use embeddings and understand why validation matters.
- When you **use RAG** over large corpora (Day 4), embeddings power the retrieval step.
- When you **interpret model behavior** (Days 3–5), knowing how attention and generation work lets you debug rather than guess.

---

## Today's Session

We build up the key ideas behind modern language models, layer by layer:

1. **Representing meaning**: from bag-of-words to word embeddings, including bias detection
2. **From static to contextual**: what a real language model can do that word vectors cannot
3. **Tokenization**: how text becomes the numbers a model processes
4. **Attention**: the mechanism that lets models decide what is relevant
5. **Generation**: next-token prediction, perplexity, and decoding strategies

Each section includes exercises that build on each other. Work through them at your own pace. Solutions are available in collapsed cells below each exercise.


## Setup

Run this cell to install dependencies. This takes ~1-2 minutes on Colab.


In [ ]:
# Install dependencies
!pip install -q gensim tiktoken transformers torch matplotlib numpy accelerate

import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

print("✓ All libraries installed and imported.")
print(f"  PyTorch: {torch.__version__}")
print(f"  Device: {'GPU ✓' if torch.cuda.is_available() else 'CPU (GPU not required for this notebook)'}")


---
# **Section 1: Representing Meaning**
*From bag-of-words to word embeddings*

---

The first question in building a language model is deceptively simple: **how do you represent the meaning of a word as a number?**

The answer to this question has evolved dramatically, and the evolution mirrors the leap from traditional NLP to modern LLMs. We will start with the simplest approach, discover its limitations, and then see how embeddings solve them.


## 1.1 Bag of Words: The Simplest Representation

The most straightforward approach: represent each document as a **count of which words appear in it**. This is called a **bag-of-words** (BoW) representation, "bag" because we throw away word order entirely.

This idea is not just a toy example. BoW underpins an entire family of methods that social scientists have used productively for decades: TF-IDF for document retrieval, topic models like LDA and STM (Roberts et al., 2014) for discovering themes in large corpora, and dictionary methods for measuring sentiment, moral foundations, or policy focus. If you have done text analysis before, chances are you have used a BoW-family method.

Let's see how it works, and where it breaks down.


In [ ]:
# Five sentences. Some are about the same topic, some are not.
sentences = [
    "The president signed the new economic policy today",            # A
    "Economic growth depends on sound fiscal policy",                # B
    "The government proposed new measures to stimulate the economy",  # C
    "The striker scored a hat-trick in the Champions League final",   # D
    "The cat sat on the warm windowsill",                            # E
]
labels = ["A: economic policy", "B: fiscal policy",
          "C: stimulate economy", "D: sports", "E: cat"]

# Build BoW vectors: count how often each word appears
all_words = set()
for s in sentences:
    all_words.update(s.lower().split())
vocab = sorted(all_words)

def sentence_to_bow(sentence, vocab):
    words = sentence.lower().split()
    counts = Counter(words)
    return np.array([counts.get(w, 0) for w in vocab], dtype=np.float64)

bow_vectors = [sentence_to_bow(s, vocab) for s in sentences]

# What does a BoW vector look like?
print(f"Vocabulary: {len(vocab)} unique words across all sentences.\n")
print(f'Sentence A: "{sentences[0]}"')
print(f"BoW vector (showing non-zero entries):")
for j, w in enumerate(vocab):
    if bow_vectors[0][j] > 0:
        print(f"  '{w}': {int(bow_vectors[0][j])}")

### Exercise 1a: Be the Algorithm

A computer using BoW decides whether two sentences are similar by checking **how many words they share.** Let's do exactly that.

**Tasks:**

1. Write a function `shared_words(s1, s2)` that returns the set of words two sentences have in common (lowercased).
2. Use it to compute the shared word count for every pair of our 5 sentences.
3. Look at the results. Which pairs does word-overlap say are most similar? Does that match your intuition?

**Hint:** Convert each sentence to a set of lowercased words with `set(s.lower().split())`, then use the `&` (intersection) operator.


In [ ]:
def shared_words(s1, s2):
    """Return the set of words shared between two sentences."""
    # YOUR CODE HERE (2 lines)
    pass

# Compare all pairs
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        shared = shared_words(sentences[i], sentences[j])
        print(f"  {labels[i]:25s} ↔ {labels[j]:25s}  "
              f"shared: {len(shared):2d}  {shared if shared else '{}'}")

In [ ]:
#@title Solution 1a
def shared_words(s1, s2):
    """Return the set of words shared between two sentences."""
    words1 = set(s1.lower().split())
    words2 = set(s2.lower().split())
    return words1 & words2

for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        shared = shared_words(sentences[i], sentences[j])
        print(f"  {labels[i]:25s} ↔ {labels[j]:25s}  "
              f"shared: {len(shared):2d}  {shared if shared else '{}'}")

print("\n→ Look at A ↔ C. Both are about economic policy, but they share")
print("  only common function words ('the', 'new'). As far as word overlap")
print("  is concerned, 'stimulate the economy' has nothing to do with")
print("  'economic policy.' That is the ceiling of counting words.")

The raw shared-word count is a crude measure: it does not account for sentence length, and it treats every shared word equally. **Cosine similarity** is the standard way to compare vectors. It measures the angle between them:

$$\text{cosine\_similarity}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|}$$

- **1.0** means the vectors point in the same direction (identical word usage)
- **0.0** means orthogonal (no overlap)

It is the principled version of what you just did: measuring how much two bags of words overlap, normalized for length.


In [ ]:
#@title Optional: Vector Algebra Refresher
# Quick visual refresher on vectors, dot products, and cosine similarity.
# Skip this if you are comfortable with linear algebra basics.

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel 1: two vectors and the angle between them
ax = axes[0]
a = np.array([3, 1])
b = np.array([1, 3])
ax.annotate("", xy=a, xytext=(0,0), arrowprops=dict(arrowstyle="->", color="steelblue", lw=2))
ax.annotate("", xy=b, xytext=(0,0), arrowprops=dict(arrowstyle="->", color="coral", lw=2))
ax.text(a[0]+0.1, a[1]+0.1, "a = [3, 1]", color="steelblue", fontsize=11)
ax.text(b[0]+0.1, b[1]+0.1, "b = [1, 3]", color="coral", fontsize=11)
angle = np.arccos(np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b)))
arc = np.linspace(0, angle, 30)
ax.plot(0.8*np.cos(arc), 0.8*np.sin(arc), 'k-', lw=1)
ax.text(0.7, 0.7, f"θ", fontsize=12)
ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, 4)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title("Two vectors and the angle θ")

# Panel 2: dot product
ax = axes[1]
ax.axis('off')
ax.text(0.05, 0.85, "Dot product:", fontsize=13, fontweight='bold', transform=ax.transAxes)
ax.text(0.05, 0.70, "a · b = a₁b₁ + a₂b₂ + ... + aₙbₙ", fontsize=12, family='monospace', transform=ax.transAxes)
ax.text(0.05, 0.55, "Example:  [3,1] · [1,3] = 3×1 + 1×3 = 6", fontsize=11, transform=ax.transAxes)
ax.text(0.05, 0.38, "Norm (length):", fontsize=13, fontweight='bold', transform=ax.transAxes)
ax.text(0.05, 0.23, "‖a‖ = √(a₁² + a₂² + ... + aₙ²)", fontsize=12, family='monospace', transform=ax.transAxes)
ax.text(0.05, 0.08, "Example:  ‖[3,1]‖ = √(9+1) = √10 ≈ 3.16", fontsize=11, transform=ax.transAxes)
ax.set_title("Key operations")

# Panel 3: cosine similarity cases
ax = axes[2]
cases = [
    (np.array([2,0]), np.array([2,0]), "Same direction\ncos = 1.0"),
    (np.array([2,0]), np.array([0,2]), "Perpendicular\ncos = 0.0"),
    (np.array([2,0]), np.array([1.6,1.2]), "Similar\ncos ≈ 0.8"),
]
colors_list = ["green", "red", "orange"]
for k, (v1, v2, label) in enumerate(cases):
    yoff = 2 - k * 1.0
    ax.annotate("", xy=(v1[0], v1[1]+yoff), xytext=(0, yoff),
                arrowprops=dict(arrowstyle="->", color="steelblue", lw=2))
    ax.annotate("", xy=(v2[0], v2[1]+yoff), xytext=(0, yoff),
                arrowprops=dict(arrowstyle="->", color="coral", lw=2))
    ax.text(2.5, yoff, label, fontsize=10, va='center', color=colors_list[k])
ax.set_xlim(-0.5, 4.5); ax.set_ylim(-0.5, 3.5)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title("Cosine similarity: three cases")

plt.tight_layout()
plt.show()

print("Cosine similarity = dot product / (norm × norm) = cos(θ)")
print("It measures the ANGLE between vectors, ignoring their length.")
print("This is why it works for comparing documents of different lengths.")

### Exercise 1b: Implement Cosine Similarity

Implement the `cosine_similarity` function using numpy, then compare all our BoW sentence pairs. Does the ranking match what you found by counting shared words?

**Hints:**
- `np.dot(a, b)` computes the dot product
- `np.linalg.norm(a)` computes the L2 norm (vector length)


In [ ]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two numpy vectors."""
    # YOUR CODE HERE (2-3 lines)
    pass

# Compare all pairs using BoW vectors
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = cosine_similarity(bow_vectors[i], bow_vectors[j])
        print(f"  {labels[i]:25s} ↔ {labels[j]:25s}  sim = {sim:.3f}")

In [ ]:
#@title Solution 1b
def cosine_similarity(a, b):
    """Compute cosine similarity between two numpy vectors."""
    dot = np.dot(a, b)
    norms = np.linalg.norm(a) * np.linalg.norm(b)
    if norms == 0:
        return 0.0
    return dot / norms

for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = cosine_similarity(bow_vectors[i], bow_vectors[j])
        print(f"  {labels[i]:25s} ↔ {labels[j]:25s}  sim = {sim:.3f}")

print("\n→ Same story as your word-counting, but normalized.")
print("  BoW correctly separates politics from sports and cats.")
print("  But A ↔ C (both about economic policy) gets low similarity.")
print("  BoW cannot see that 'fiscal' ≈ 'economic' or")
print("  'stimulate the economy' ≈ 'economic growth'.")

**Where BoW hits its ceiling:** All BoW-family methods (raw counts, TF-IDF, topic models, dictionaries) represent meaning through **which words appear.** They work when different concepts use different vocabulary.

But sentences A and C are both about economic policy, yet they share almost no words. BoW does not know that "fiscal" is related to "economic," or that "stimulate the economy" means the same thing as "economic growth." Every word is equally different from every other word: "cat" is as far from "dog" as it is from "democracy."

We need a representation where **similar words are close together.** That is what embeddings give us.


## 1.2 Word Embeddings: Meaning as Geometry

The breakthrough idea (Mikolov et al., 2013): instead of representing words as sparse count vectors, **learn a dense vector for each word from its context.** Words that appear in similar contexts get similar vectors.

This is the **distributional hypothesis**: "you shall know a word by the company it keeps" (Firth, 1957).

Let's load pre-trained word embeddings and see what this buys us:


In [ ]:
import gensim.downloader as api

# Load GloVe embeddings (50-dimensional, trained on Wikipedia + Gigaword)
# This downloads ~66MB on first run
print("Loading GloVe embeddings (this takes ~30 seconds the first time)...")
glove = api.load("glove-wiki-gigaword-50")
print(f"✓ Loaded {len(glove)} word vectors, each with {glove.vector_size} dimensions")

# What does a word vector look like?
print(f"\nVector for 'democracy' (first 10 of 50 dimensions):")
print(glove['democracy'][:10].round(3))


In [ ]:
# The magic: similar words are nearby in this space
print("Words most similar to 'democracy':")
for word, score in glove.most_similar('democracy', topn=8):
    print(f"  {word:20s} {score:.3f}")

print("\nWords most similar to 'parliament':")
for word, score in glove.most_similar('parliament', topn=8):
    print(f"  {word:20s} {score:.3f}")


### Exercise 1c: Exploring the Embedding Space

Word embeddings encode **relationships as directions.** The most famous example: "king − man + woman ≈ queen." This works because the vector from "man" to "king" captures something like "royalty," and adding that same direction to "woman" lands near "queen."

**Part 1:** Use `glove.most_similar(positive=[...], negative=[...])` to test these analogies:
- king − man + woman = ?
- paris − france + germany = ?
- Come up with 2 of your own (at least one that works, one that fails).

**Part 2:** Pick a political concept (e.g., "terrorism", "immigration", "freedom") and look at its nearest neighbors. What does the embedding space "think" this concept means?


In [ ]:
# PART 1: Word analogies
# Example syntax: glove.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)

# YOUR CODE HERE: test at least 4 analogies



# PART 2: Pick a political concept, explore its neighborhood
# YOUR CODE HERE



In [ ]:
#@title Solution 1c
# PART 1
print("king - man + woman =")
for word, score in glove.most_similar(positive=['king', 'woman'], negative=['man'], topn=5):
    print(f"  {word:20s} {score:.3f}")

print("\nparis - france + germany =")
for word, score in glove.most_similar(positive=['paris', 'germany'], negative=['france'], topn=5):
    print(f"  {word:20s} {score:.3f}")

print("\ndemocracy - freedom + oppression =")
for word, score in glove.most_similar(positive=['democracy', 'oppression'], negative=['freedom'], topn=5):
    print(f"  {word:20s} {score:.3f}")

print("\nsenator - america + britain =")
for word, score in glove.most_similar(positive=['senator', 'britain'], negative=['america'], topn=3):
    print(f"  {word:20s} {score:.3f}")

# PART 2
print("\n--- Neighborhood of 'terrorism' ---")
for word, score in glove.most_similar('terrorism', topn=10):
    print(f"  {word:20s} {score:.3f}")


## 1.3 What Embeddings Encode, Including Bias

Since embeddings learn from human text, they absorb human **biases and stereotypes.** This is simultaneously a problem (if you use embeddings in a pipeline) and a research tool (if you want to *measure* bias).

Caliskan et al. (2017) showed that word embeddings replicate every implicit bias measured in humans. Kozlowski et al. (2019) used embedding geometry to map cultural dimensions like class and gender.

The technique: define a **semantic axis** using pairs of anchor words (e.g., "man"/"woman"), then project other words onto that axis to see where they fall.


In [ ]:
def semantic_axis(glove, positive_anchors, negative_anchors):
    """Create a semantic direction from anchor word pairs. Returns a unit vector."""
    pos = np.mean([glove[w] for w in positive_anchors], axis=0)
    neg = np.mean([glove[w] for w in negative_anchors], axis=0)
    direction = pos - neg
    return direction / np.linalg.norm(direction)

def project_words(glove, words, axis):
    """Project words onto a semantic axis. Returns sorted (word, score) pairs."""
    results = []
    for w in words:
        if w in glove:
            score = np.dot(glove[w], axis)
            results.append((w, score))
    return sorted(results, key=lambda x: x[1])

# Demo: gender axis
gender_axis = semantic_axis(glove, ['woman', 'she', 'her'], ['man', 'he', 'him'])

occupations = ['nurse', 'doctor', 'teacher', 'engineer', 'secretary',
               'professor', 'soldier', 'dancer', 'programmer', 'receptionist',
               'scientist', 'librarian', 'pilot', 'therapist']

print("Occupations projected onto gender axis (← male | female →)\n")
for word, score in project_words(glove, occupations, gender_axis):
    bar = "█" * int(abs(score) * 15)
    direction = "→" if score > 0 else "←"
    print(f"  {word:15s} {score:+.3f}  {direction} {bar}")


### Exercise 1d: Mapping Cultural Dimensions

**Your turn.** Define your own semantic axes and project relevant words onto them.

**Tasks:**
1. Create a **political ideology axis** (e.g., "conservative"/"liberal" as anchors). Project political concepts and institutions onto it.
2. Create a **valence axis** (good/bad). Project political actors, countries, or concepts onto it. What biases do you find?

**Think about:** What do the results tell you about the text these embeddings were trained on (Wikipedia + news)? How could this be used as a measurement tool in research?


In [ ]:
# TASK 1: Political ideology axis
# Define your anchors, then project relevant words

# YOUR CODE HERE
# ideology_axis = semantic_axis(glove, [...], [...])
# political_words = [...]
# for word, score in project_words(glove, political_words, ideology_axis):
#     print(f"  {word:20s} {score:+.3f}")


# TASK 2: Valence axis
# YOUR CODE HERE



In [ ]:
#@title Solution 1d
# TASK 1: Political ideology axis
ideology_axis = semantic_axis(
    glove,
    ['liberal', 'progressive', 'democratic'],
    ['conservative', 'traditional', 'republican']
)

political_words = ['congress', 'senate', 'regulation', 'military', 'welfare',
                   'tax', 'immigration', 'healthcare', 'trade', 'abortion',
                   'gun', 'climate', 'police', 'union', 'church', 'university']

print("Political concepts on ideology axis (← conservative | liberal →)\n")
for word, score in project_words(glove, political_words, ideology_axis):
    bar = "█" * int(abs(score) * 12)
    direction = "→" if score > 0 else "←"
    print(f"  {word:15s} {score:+.3f}  {direction} {bar}")

# TASK 2: Valence axis
print("\n" + "="*50)
valence_axis = semantic_axis(
    glove,
    ['good', 'wonderful', 'excellent', 'positive'],
    ['bad', 'terrible', 'awful', 'negative']
)

concepts = ['democracy', 'terrorism', 'freedom', 'war', 'peace',
            'corruption', 'justice', 'poverty', 'education', 'violence',
            'china', 'america', 'russia', 'europe', 'africa']

print("\nConcepts on valence axis (← negative | positive →)\n")
for word, score in project_words(glove, concepts, valence_axis):
    bar = "█" * int(abs(score) * 12)
    direction = "→" if score > 0 else "←"
    print(f"  {word:15s} {score:+.3f}  {direction} {bar}")

print("\n→ Notice how embeddings encode evaluative associations from the")
print("  training corpus. This is both a bias concern and a research")
print("  opportunity (cf. Caliskan et al., 2017; Kozlowski et al., 2019).")


**Key takeaway from Section 1:** Words can be represented as vectors in a meaningful geometric space. Similar words are close together, relationships are directions, and the geometry encodes rich (sometimes problematic) cultural information.

But there is a fundamental limitation we have not addressed yet. Consider the word **"bank"**:
- *"I walked along the river bank"*
- *"I deposited money at the bank"*

In GloVe, "bank" has **one vector** regardless of context. It cannot distinguish these two meanings. This is true for every word: the same representation no matter what sentence it appears in.

What if a model could produce **different representations of the same word depending on context?** That is exactly what modern language models do.

---


---
# **Section 2: From Static Vectors to Language Understanding**
*What a real language model can do, and how it works*

---

GloVe gives every word a fixed vector. A modern language model does something much more powerful: it **reads the whole sequence** and produces representations that depend on context.

Let's see this in action. We will load a small, modern language model, **Qwen3-0.6B** (released in 2025, 600 million parameters), and look at what it actually does.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load Qwen3-0.6B, a small but capable modern model
print("Loading Qwen3-0.6B (this may take 1-2 minutes the first time)...")
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
qwen_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    torch_dtype=torch.float32,
    output_attentions=True,
)
qwen_model.eval()

n_params = sum(p.numel() for p in qwen_model.parameters())
print(f"✓ Loaded Qwen3-0.6B: {n_params/1e6:.0f}M parameters")
print(f"  Architecture: {qwen_model.config.num_hidden_layers} layers, "
      f"{qwen_model.config.num_attention_heads} attention heads")
print(f"  Vocabulary: {qwen_tokenizer.vocab_size:,} tokens")
print(f"  Released: 2025 (state-of-the-art small model)")


## 2.1 Next-Token Prediction: The Core Idea

Every language model, from GPT-2 (2019, 124M params) to Claude and GPT-4 (hundreds of billions), does the same fundamental thing: **given a sequence of text, predict what comes next.**

The model outputs a probability distribution over its entire vocabulary for the next token. Let's look inside:


In [ ]:
def show_next_token_probs(model, tokenizer, text, top_k=15):
    """Show the model's top predicted next tokens and their probabilities."""
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)

    # Get logits for the LAST token position → what comes next?
    logits = outputs.logits[0, -1, :]  # shape: (vocab_size,)
    probs = torch.softmax(logits, dim=-1)

    top_probs, top_ids = probs.topk(top_k)

    print(f"Prompt: '{text}'")
    print(f"\nTop {top_k} predicted next tokens:\n")
    for prob, idx in zip(top_probs, top_ids):
        token_text = tokenizer.decode([idx.item()])
        bar = "█" * int(prob.item() * 80)
        print(f"  '{token_text}'  {'':<10s} {prob.item():.4f}  {bar}")
    return probs


# A factual completion: the model should be confident
print("="*60)
show_next_token_probs(qwen_model, qwen_tokenizer, "The capital of France is")

# An open-ended prompt: many plausible continuations
print("\n" + "="*60)
show_next_token_probs(qwen_model, qwen_tokenizer, "The government should")

# A grammatical constraint: the model knows syntax
print("\n" + "="*60)
show_next_token_probs(qwen_model, qwen_tokenizer, "She was walking her")


Notice three things:
1. **Factual knowledge:** The model is highly confident that "Paris" follows "The capital of France is"
2. **Open-ended understanding:** For "The government should", the model distributes probability across many plausible policy verbs
3. **Grammar:** After "She was walking her", the model predicts nouns (dog, cat...): it has learned syntax

All of this from a single objective: predict the next token. A model that does this well must implicitly learn grammar, facts, reasoning, and much more.


## 2.2 Beyond Static Embeddings: Contextual Understanding

Here is something GloVe cannot do. Watch how the model's predictions change based on context, showing that it *understands* words differently depending on their surroundings:


In [ ]:
# The word "bank" in different contexts
print("Context matters: watch how predictions change:\n")

print("="*60)
show_next_token_probs(qwen_model, qwen_tokenizer,
    "I sat by the river bank and watched the", top_k=8)

print("\n" + "="*60)
show_next_token_probs(qwen_model, qwen_tokenizer,
    "I went to the bank to deposit my", top_k=8)

print("\n" + "="*60)
print("Same word, completely different understanding.\n")
print("In GloVe, 'bank' has ONE vector. Here, the model builds a")
print("DIFFERENT internal representation of 'bank' in each sentence,")
print("informed by every other word in the context.")


In [ ]:
# Another striking example: the model tracks long-range dependencies
print("Long-range understanding:\n")

print("="*60)
show_next_token_probs(qwen_model, qwen_tokenizer,
    "The scientist who had been studying climate change for decades finally published her", top_k=8)

print("\n" + "="*60)
show_next_token_probs(qwen_model, qwen_tokenizer,
    "The chef who had been perfecting his recipe for decades finally published his", top_k=8)

print("\n→ The model connects 'published' back to 'scientist'/'chef' across")
print("  many intervening words. Static embeddings cannot do this.")


## 2.3 What Makes This Possible?

So how does a language model go from static word vectors to this rich contextual understanding? Three key ingredients:

1. **Tokenization**: how text gets converted into the numbers the model processes
2. **Attention**: the mechanism that lets each word "look at" every other word to build context-dependent representations
3. **Scale**: parameters, data, and compute that turn these mechanisms into genuine understanding

Let's unpack each of these.

---


---
# **Section 3: How Models See Text**
*Tokenization: from strings to numbers*

---

We just saw Qwen3 predict next tokens. But what is a "token"? The model does not see words. It sees **tokens**, which are integer IDs mapped to subword units.

Why not just split on spaces? Three reasons:
1. **Vocabulary size**: English has millions of word forms. A model cannot have a separate vector for every one.
2. **Unknown words**: what happens when the model encounters a word it has never seen?
3. **Morphology**: "running", "runs", "ran" are related. A good tokenizer should capture this.


### Exercise 2a: Build a Word-Level Tokenizer

Let's build the simplest possible tokenizer to understand the core mechanics, and discover the fundamental problem.

**Implement** a `SimpleTokenizer` class with:
- `__init__(self, text)`: Build a vocabulary from training text. Add a special `<UNK>` token for unknown words.
- `encode(self, text)`: Convert text → list of token IDs
- `decode(self, token_ids)`: Convert token IDs → text

**Hints:**
- Use `text.split()` to get words
- Use a dictionary to map words → IDs and vice versa
- `<UNK>` should get ID 0


In [ ]:
class SimpleTokenizer:
    def __init__(self, text):
        """Build vocabulary from training text."""
        # YOUR CODE HERE
        # 1. Split text into words (lowercased)
        # 2. Get unique words
        # 3. Create word_to_id dict (with <UNK> at id 0)
        # 4. Create id_to_word dict (reverse mapping)
        pass

    def encode(self, text):
        """Convert text to a list of token IDs."""
        # YOUR CODE HERE: use <UNK> id for unknown words
        pass

    def decode(self, token_ids):
        """Convert token IDs back to text."""
        # YOUR CODE HERE
        pass


# Test it:
training_text = """The president met with the prime minister to discuss economic policy.
The minister announced new trade agreements with European partners.
Economic growth remained strong despite global uncertainty."""

tok = SimpleTokenizer(training_text)
print(f"Vocabulary size: {len(tok.word_to_id)}")
print(f"\nSample mappings:")
for word in ['the', 'president', 'economic', 'policy.']:
    if word in tok.word_to_id:
        print(f"  '{word}' → {tok.word_to_id[word]}")

# Encode and decode
test = "The president announced new economic policy"
ids = tok.encode(test)
print(f"\nEncode: '{test}'")
print(f"  → {ids}")
print(f"Decode: {tok.decode(ids)}")

# Now try something the tokenizer has NOT seen:
unseen = "The senator proposed immigration reform"
ids_unseen = tok.encode(unseen)
print(f"\nEncode unseen words: '{unseen}'")
print(f"  → {ids_unseen}")
print(f"Decode: {tok.decode(ids_unseen)}")
print("\n⚠️  What happened to words the tokenizer has not seen?")


In [ ]:
#@title Solution 2a
class SimpleTokenizer:
    def __init__(self, text):
        """Build vocabulary from training text."""
        words = text.lower().split()
        unique_words = sorted(set(words))
        self.word_to_id = {'<UNK>': 0}
        for i, word in enumerate(unique_words):
            self.word_to_id[word] = i + 1
        self.id_to_word = {v: k for k, v in self.word_to_id.items()}

    def encode(self, text):
        """Convert text to a list of token IDs."""
        return [self.word_to_id.get(w, 0) for w in text.lower().split()]

    def decode(self, token_ids):
        """Convert token IDs back to text."""
        return ' '.join(self.id_to_word.get(id, '<UNK>') for id in token_ids)


# Test it:
training_text = """The president met with the prime minister to discuss economic policy.
The minister announced new trade agreements with European partners.
Economic growth remained strong despite global uncertainty."""

tok = SimpleTokenizer(training_text)
print(f"Vocabulary size: {len(tok.word_to_id)}")
print(f"\nSample mappings:")
for word in ['the', 'president', 'economic', 'policy.']:
    if word in tok.word_to_id:
        print(f"  '{word}' → {tok.word_to_id[word]}")

test = "The president announced new economic policy"
ids = tok.encode(test)
print(f"\nEncode: '{test}'")
print(f"  → {ids}")
print(f"Decode: {tok.decode(ids)}")

unseen = "The senator proposed immigration reform"
ids_unseen = tok.encode(unseen)
print(f"\nEncode unseen words: '{unseen}'")
print(f"  → {ids_unseen}")
print(f"Decode: {tok.decode(ids_unseen)}")
print("\n→ 'senator', 'proposed', 'immigration', 'reform' all became <UNK>.")
print("  This is the fundamental problem with word-level tokenization.")


**The out-of-vocabulary (OOV) problem:** Our word-level tokenizer loses *all information* about unseen words. In social science research, where you analyze novel texts with domain-specific vocabulary, this is catastrophic.

## 3.2 Byte-Pair Encoding: The Real Solution

Modern language models use **subword tokenization.** The most common algorithm is **Byte-Pair Encoding (BPE)** (Sennrich et al., 2016):

1. Start with individual characters as your vocabulary
2. Find the most frequent pair of adjacent tokens in the training data
3. Merge that pair into a new token
4. Repeat until you reach your desired vocabulary size

The result:
- **Common words** ("the", "and") → single token
- **Rare words** ("democratization") → split into subwords ("demo" + "crat" + "ization")
- **Nothing is ever unknown**: in the worst case, the model falls back to individual characters

Let's compare our toy tokenizer with a real one. We will use the one inside Qwen3 (the model we loaded in Section 2):


In [ ]:
# Let's use the Qwen3 tokenizer we already loaded
print(f"Qwen3 vocabulary size: {qwen_tokenizer.vocab_size:,} tokens\n")

# Compare with our word tokenizer
text = "The senator proposed immigration reform"

print(f"Our word tokenizer:  {tok.encode(text)}")
print(f"  Decoded: '{tok.decode(tok.encode(text))}'\n")

qwen_tokens = qwen_tokenizer.encode(text)
print(f"Qwen3 BPE tokenizer: {qwen_tokens}")
print(f"  Decoded: '{qwen_tokenizer.decode(qwen_tokens)}'")
print(f"  Token-by-token: {[qwen_tokenizer.decode([t]) for t in qwen_tokens]}")
print(f"\n→ No information lost! Every word is represented, even ones the")
print(f"  tokenizer has never seen as a complete word.")


### Exercise 2b: Tokenizer Detective

Tokenization has **real consequences** for model behavior. Investigate how BPE handles different inputs.

**Tasks:**

1. **The "strawberry" test:** How many tokens does "strawberry" need? Try "Strawberry" and "STRAWBERRY" too. Why might a model struggle to count letters in "strawberry"?

2. **Multilingual inequality:** Tokenize the same sentence in English and 1-2 other languages. Compare token counts. What does this mean for cost and performance?

3. **Social science terms:** Tokenize domain-specific terms ("gerrymandering", "filibuster", "democratization") and names from different cultures. Any surprises?

4. **Numbers:** How does the tokenizer handle "1000" vs "1,000" vs "$1,000.00"?


In [ ]:
def show_tokens(text, tokenizer=qwen_tokenizer):
    """Show how a tokenizer splits text."""
    tokens = tokenizer.encode(text)
    pieces = [tokenizer.decode([t]) for t in tokens]
    print(f"'{text}'")
    print(f"  {len(tokens)} tokens: {pieces}\n")

# TASK 1: The "strawberry" test
# YOUR CODE HERE


# TASK 2: Multilingual comparison
# YOUR CODE HERE
# Try: "The United Nations discussed climate change" in English + other languages


# TASK 3: Social science terms and names
# YOUR CODE HERE


# TASK 4: Numbers
# YOUR CODE HERE



In [ ]:
#@title Solution 2b
def show_tokens(text, tokenizer=qwen_tokenizer):
    tokens = tokenizer.encode(text)
    pieces = [tokenizer.decode([t]) for t in tokens]
    print(f"'{text}'")
    print(f"  {len(tokens)} tokens: {pieces}\n")

# TASK 1
print("=== The 'strawberry' test ===")
show_tokens("strawberry")
show_tokens("Strawberry")
show_tokens("STRAWBERRY")
print("→ The model sees subword pieces, not individual letters.")
print("  Counting 'r's requires reasoning over subword boundaries.\n")

# TASK 2
print("=== Multilingual inequality ===")
show_tokens("The United Nations discussed climate change")
show_tokens("联合国讨论了气候变化")       # Chinese
show_tokens("Las Naciones Unidas discutieron el cambio climático")  # Spanish
show_tokens("الأمم المتحدة ناقشت تغير المناخ")  # Arabic
print("→ Same meaning, different token counts. More tokens = higher cost,")
print("  shorter effective context, potentially worse performance.\n")

# TASK 3
print("=== Social science terms ===")
show_tokens("gerrymandering")
show_tokens("filibuster")
show_tokens("democratization")
show_tokens("Bolsonaro")
show_tokens("Xi Jinping")
show_tokens("Zelenskyy")
print("→ Rare names and technical terms get fragmented.\n")

# TASK 4
print("=== Numbers ===")
show_tokens("1000")
show_tokens("1,000")
show_tokens("$1,000.00")
show_tokens("2024-01-15")
show_tokens("January 15, 2024")
print("→ Different number formats produce different tokenizations.")


**Key takeaway from Section 3:** Tokenization is the bridge between human text and model computation. BPE solves the unknown-word problem elegantly but introduces its own artifacts: subword splits, multilingual inequality, and number fragmentation. When working with LLMs, **the tokenizer is part of your measurement instrument.**

---


---
# **Section 4: Attention: How Models Decide What Matters**
*The mechanism at the heart of every modern language model*

---

We have seen that a language model:
- Converts text into token IDs (Section 3)
- Turns each token ID into a vector via an embedding layer (like Section 1, but learned)
- Somehow produces context-dependent predictions (Section 2)

The "somehow" is **attention**, the key innovation in the Transformer architecture (Vaswani et al., 2017). Before we look at the math, let's build intuition for what attention does by doing it ourselves.


### Exercise 3a: Be the Attention Mechanism

When you read a sentence and predict what comes next, you do not weigh every word equally. Some words matter more than others for the prediction.

**Task:** For each sentence below, a word is missing (marked `___`). Your job is to assign an **importance weight** (between 0 and 1) to every preceding word, reflecting how much that word helps you predict the blank. Weights should sum to 1.

Do this by filling in the `weights` lists in the code below. Think carefully: which words would YOU look at to make the prediction?


In [ ]:
# Assign importance weights to each word for predicting the blank.
# Weights must be between 0 and 1, and sum to 1 for each sentence.

# Sentence 1: "The prime minister of the United Kingdom met with the ___"
words_1 = ["The", "prime", "minister", "of", "the", "United", "Kingdom", "met", "with", "the"]
weights_1 = [0.0] * 10  # YOUR WEIGHTS HERE: e.g. [0.0, 0.3, 0.1, ...]

# Sentence 2: "The doctor told the patient that she should ___"
words_2 = ["The", "doctor", "told", "the", "patient", "that", "she", "should"]
weights_2 = [0.0] * 8  # YOUR WEIGHTS HERE

# Sentence 3: "After years of conflict, the two nations finally signed a ___"
words_3 = ["After", "years", "of", "conflict", "the", "two", "nations", "finally", "signed", "a"]
weights_3 = [0.0] * 10  # YOUR WEIGHTS HERE

# --- Visualize your weights ---
fig, axes = plt.subplots(3, 1, figsize=(12, 7))
for ax, words, weights, sent_num in zip(axes, [words_1, words_2, words_3],
                                         [weights_1, weights_2, weights_3],
                                         [1, 2, 3]):
    colors = plt.cm.Blues(np.array(weights) / max(max(weights), 0.01))
    bars = ax.bar(range(len(words)), weights, color=colors, edgecolor='steelblue')
    ax.set_xticks(range(len(words)))
    ax.set_xticklabels(words, fontsize=10)
    ax.set_ylabel("Weight")
    ax.set_title(f"Sentence {sent_num}: {'  '.join(words)} ___")
    ax.set_ylim(0, max(max(weights), 0.01) * 1.3)

plt.tight_layout()
plt.show()

# Check: do weights sum to ~1?
for i, (w, name) in enumerate([(weights_1, "S1"), (weights_2, "S2"), (weights_3, "S3")]):
    total = sum(w)
    status = "✓" if abs(total - 1.0) < 0.05 else f"⚠ (sums to {total:.2f}, should be ~1.0)"
    print(f"  {name} weight sum: {total:.2f} {status}")

In [ ]:
#@title Solution 3a
# There is no single "correct" answer! But here is one reasonable assignment.

words_1 = ["The", "prime", "minister", "of", "the", "United", "Kingdom", "met", "with", "the"]
weights_1 = [0.0,   0.15,   0.15,     0.0,  0.0,  0.05,     0.05,      0.25,  0.25,  0.10]
# "met with the ___" needs: who is being met? → "prime minister" matters
# "met with" signals someone is about to be named → high weight

words_2 = ["The", "doctor", "told", "the", "patient", "that", "she", "should"]
weights_2 = [0.0,  0.25,    0.10,  0.0,   0.25,     0.0,    0.10,  0.30]
# "should ___" → what kind of action? "doctor" and "patient" set the domain
# "should" is the strongest signal for predicting a verb

words_3 = ["After", "years", "of", "conflict", "the", "two", "nations", "finally", "signed", "a"]
weights_3 = [0.0,    0.05,   0.0,  0.20,      0.0,   0.10,  0.15,     0.05,      0.35,     0.10]
# "signed a ___" → what do nations sign after conflict? "signed" and "conflict" dominate

fig, axes = plt.subplots(3, 1, figsize=(12, 7))
for ax, words, weights, sent_num in zip(axes, [words_1, words_2, words_3],
                                         [weights_1, weights_2, weights_3],
                                         [1, 2, 3]):
    colors = plt.cm.Blues(np.array(weights) / max(weights))
    ax.bar(range(len(words)), weights, color=colors, edgecolor='steelblue')
    ax.set_xticks(range(len(words)))
    ax.set_xticklabels(words, fontsize=10)
    ax.set_ylabel("Weight")
    ax.set_title(f"Sentence {sent_num}: {'  '.join(words)} ___")
    ax.set_ylim(0, max(weights) * 1.3)
plt.tight_layout()
plt.show()

print("\n→ You just computed attention. You looked at every word in the")
print("  sequence and decided how much each one matters for predicting")
print("  what comes next. That is exactly what the attention mechanism does.")

### Context Changes Everything

Notice something deeper: the weights you assign depend on **what you are trying to predict.** The same word can demand completely different attention patterns depending on context.

Consider the word "bank" in these two sentences:


In [ ]:
# Same word, different context, different attention patterns
print("Where does your attention go to understand 'bank'?\n")

s1_words = ["I", "sat", "by", "the", "river", "bank"]
s2_words = ["I", "went", "to", "the", "bank", "to", "deposit", "my", "money"]

print(f"  Sentence 1: {' '.join(s1_words)}")
print(f"  → Which words tell you 'bank' means a riverbank?")
print(f"     Probably 'river' (and 'sat by')\n")

print(f"  Sentence 2: {' '.join(s2_words)}")
print(f"  → Which words tell you 'bank' means a financial institution?")
print(f"     Probably 'deposit' and 'money'\n")

print("→ The SAME word requires DIFFERENT attention patterns depending on context.")
print("  This is exactly what we saw in Section 2: the model builds a different")
print("  internal representation of 'bank' in each sentence. Attention is how.")

### Why One Set of Weights Is Not Enough

Now consider a more complex sentence:

*"The doctor from London told the patient that she should ___"*

To predict the blank, you need to answer several questions **simultaneously**:
- **Who** should do something? → look at "doctor" and "patient"
- **What kind** of action? → look at "doctor," "told," medical context
- **Who** does "she" refer to? → look at "doctor" or "patient" (grammar/context)

Try to capture all three with a single set of weights. You will find yourself compromising: putting weight on "doctor" for both the who-question and the what-kind question, with no way to separate those roles.

This is why transformers use **multi-head attention**: multiple sets of weights running in parallel, each one asking a different question about the sequence. One head might track syntax (subject-verb), another might track coreference ("she" → "patient"), another might focus on local context (the word right before the blank).


## 4.2 The Attention Mechanism: Formalizing Your Intuition

What you did by hand, the model does with linear algebra. Here is the recipe:

1. Each token vector is transformed into three vectors: **Query (Q)**, **Key (K)**, **Value (V)**
2. The **Query** asks: "what am I looking for?" (your question about the blank)
3. Each **Key** says: "here is what I contain" (what each word offers)
4. **Attention scores** = how well each Query matches each Key (dot product)
5. Scores are **scaled** (by $\sqrt{d_k}$) and passed through **softmax**, giving weights that sum to 1
6. Output = weighted sum of **Values** (the information you extract)

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

One critical detail: **causal masking.** Since the model predicts the *next* token, each token can only attend to itself and earlier tokens. No peeking at the future.


### Exercise 3b: Compute Attention with Math

Now let's formalize what you did by hand. We have 4 tokens with pre-defined Q, K, V matrices (in a real model, these are learned). Work through each step.


In [ ]:
# Setup: 4 tokens, each with 3-dimensional Q, K, V vectors
tokens = ["The", "cat", "sat", "down"]
d_k = 3  # dimension of key vectors

Q = torch.tensor([
    [1.0, 0.1, 0.0],   # "The" query
    [0.2, 1.0, 0.3],   # "cat" query
    [0.0, 0.3, 1.0],   # "sat" query
    [0.1, 0.8, 0.2],   # "down" query
])
K = torch.tensor([
    [0.8, 0.0, 0.1],   # "The" key
    [0.1, 0.9, 0.2],   # "cat" key
    [0.0, 0.2, 0.9],   # "sat" key
    [0.2, 0.7, 0.3],   # "down" key
])
V = torch.tensor([
    [1.0, 0.0, 0.0],   # "The" value
    [0.0, 1.0, 0.0],   # "cat" value
    [0.0, 0.0, 1.0],   # "sat" value
    [0.5, 0.5, 0.0],   # "down" value
])

# STEP 1: Compute raw attention scores (Q · Kᵀ)
# How well does each token's query match each token's key?
# YOUR CODE HERE
# scores = ...
# print("Step 1 — Raw scores (Q·Kᵀ):"); print(scores.round(decimals=3))


# STEP 2: Scale by √d_k (prevents softmax from becoming too sharp)
# YOUR CODE HERE
# scaled = ...
# print(f"\nStep 2 — Scaled (÷ √{d_k}):"); print(scaled.round(decimals=3))


# STEP 3: Causal mask (each token can only see itself and earlier tokens)
# Set upper-triangle to -inf so softmax gives them ~0 weight.
# Hint: torch.triu(torch.ones(4,4), diagonal=1).bool()
# Hint: tensor.masked_fill(mask, float('-inf'))
# YOUR CODE HERE
# mask = ...
# masked = ...
# print("\nStep 3 — Masked:"); print(masked.round(decimals=3))


# STEP 4: Softmax (each row sums to 1: these are your attention weights)
# YOUR CODE HERE
# weights = ...
# print("\nStep 4 — Attention weights:"); print(weights.round(decimals=3))


# STEP 5: Weighted sum of values (extract information)
# YOUR CODE HERE
# output = ...
# print("\nStep 5 — Output:"); print(output.round(decimals=3))


# INTERPRET: Which token does "down" (row 3) attend to most? Why?


In [ ]:
#@title Solution 3b
# STEP 1
scores = Q @ K.T
print("Step 1 — Raw scores (Q·Kᵀ):")
print(scores.round(decimals=3))

# STEP 2
scaled = scores / (d_k ** 0.5)
print(f"\nStep 2 — Scaled (÷ √{d_k} = {d_k**0.5:.2f}):")
print(scaled.round(decimals=3))

# STEP 3
mask = torch.triu(torch.ones(4, 4), diagonal=1).bool()
masked = scaled.masked_fill(mask, float('-inf'))
print("\nStep 3 — Masked (future = -inf):")
print(masked.round(decimals=3))

# STEP 4
weights = torch.softmax(masked, dim=-1)
print("\nStep 4 — Attention weights (rows sum to 1):")
print(weights.round(decimals=3))

# STEP 5
output = weights @ V
print("\nStep 5 — Output:")
print(output.round(decimals=3))

# Interpretation
print("\n--- Interpretation ---")
print(f"'down' (row 3) attends to:")
for idx, t in enumerate(tokens):
    print(f"  '{t}': {weights[3,idx]:.3f}")
print("→ 'down' attends most to 'cat', its semantic anchor.")
print("  Compare this to the weights you assigned by hand in Exercise 3a!")

# Visualize
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(weights.numpy(), cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_xticklabels(tokens)
ax.set_yticks(range(4)); ax.set_yticklabels(tokens)
ax.set_xlabel("Attends to (Key)"); ax.set_ylabel("Token (Query)")
ax.set_title("Attention Weights (Causal Masked)")
for i in range(4):
    for j in range(4):
        v = weights[i, j].item()
        if v > 0.01:
            ax.text(j, i, f"{v:.2f}", ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

### Exercise 3c: Attention in a Real Model

Let's see what attention looks like at scale. We will load **GPT-2** (124M parameters, 12 layers × 12 heads = 144 attention patterns) and visualize what its heads have learned.


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("Loading GPT-2...")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2", output_attentions=True)
gpt2_model.eval()
print(f"✓ GPT-2: {sum(p.numel() for p in gpt2_model.parameters())/1e6:.0f}M params, "
      f"{gpt2_model.config.n_layer} layers × {gpt2_model.config.n_head} heads")

In [ ]:
def get_attention(model, tokenizer, text):
    """Extract attention weights from a model."""
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    attn = torch.stack(outputs.attentions).squeeze(1)
    tokens = [tokenizer.decode([t]) for t in inputs['input_ids'][0]]
    return attn, tokens

def plot_attention(attn, tokens, layer, head=None, figsize=(8, 6)):
    """Plot attention for a layer. If head=None, show all heads."""
    if head is not None:
        data = attn[layer, head].numpy()
        fig, ax = plt.subplots(figsize=figsize)
        im = ax.imshow(data, cmap='Blues', vmin=0)
        ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45, ha='right')
        ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
        ax.set_title(f"Layer {layer}, Head {head}")
        ax.set_xlabel("Attends to"); ax.set_ylabel("Token")
        plt.colorbar(im, ax=ax)
    else:
        n_heads = attn.shape[1]
        cols = 4; rows = (n_heads + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
        axes_flat = axes.flatten()
        for h in range(n_heads):
            axes_flat[h].imshow(attn[layer, h].numpy(), cmap='Blues', vmin=0)
            axes_flat[h].set_title(f"Head {h}", fontsize=9)
            axes_flat[h].set_xticks(range(len(tokens)))
            axes_flat[h].set_xticklabels(tokens, rotation=45, ha='right', fontsize=7)
            axes_flat[h].set_yticks(range(len(tokens)))
            axes_flat[h].set_yticklabels(tokens, fontsize=7)
        for h in range(n_heads, len(axes_flat)):
            axes_flat[h].axis('off')
        fig.suptitle(f"All Heads, Layer {layer}", fontsize=12)
    plt.tight_layout()
    plt.show()

print("✓ Visualization functions ready.")

**Your turn.** Investigate what GPT-2's attention heads have learned.

**Tasks:**

1. Visualize **all 12 heads in layer 0** (the first layer). Can you find heads with specific patterns? Look for:
   - A head that mostly attends to the **previous token**
   - A head that attends to the **first token**
   - A head with a **spread-out** (diffuse) pattern

2. Compare with **layer 11** (the last layer). How do the patterns differ?

3. Try a sentence with a **long-range dependency** (e.g., "The politician who campaigned on healthcare reform won the election"). Can you find a head connecting "won" back to "politician"?


In [ ]:
text = "The prime minister of the United Kingdom met with the president of France"
attn, tokens = get_attention(gpt2_model, gpt2_tokenizer, text)
print(f"Shape: {attn.shape} = (layers, heads, tokens, tokens)")
print(f"Tokens: {tokens}")

# TASK 1: Layer 0
# YOUR CODE HERE
# plot_attention(attn, tokens, layer=0)

# TASK 2: Layer 11
# YOUR CODE HERE
# plot_attention(attn, tokens, layer=11)

# TASK 3: Long-range dependency
# YOUR CODE HERE


In [ ]:
#@title Solution 3c
text = "The prime minister of the United Kingdom met with the president of France"
attn, tokens = get_attention(gpt2_model, gpt2_tokenizer, text)

print("=== Layer 0: early patterns ===")
print("Look for: previous-token heads, first-token heads, local patterns\n")
plot_attention(attn, tokens, layer=0)

print("\n=== Layer 11: late patterns ===")
print("Look for: semantic connections, long-range attention\n")
plot_attention(attn, tokens, layer=11)

text2 = "The politician who campaigned on healthcare reform won the election"
attn2, tok2 = get_attention(gpt2_model, gpt2_tokenizer, text2)
print("\n=== Long-range dependency ===")
print("Can you find a head connecting 'won' to 'politician'?\n")
plot_attention(attn2, tok2, layer=11)

#### Stretch 3d: How Attention Reach Changes Across Layers

Compute the **average attention distance** for each layer: how far back does each token typically look? Early layers tend to be local, later layers reach further.


In [ ]:
# STRETCH: Average attention distance per layer
# For each layer, compute how far back tokens attend on average.
# Distance between token i attending to token j = |i - j|
# Weight by attention probability, average over heads and positions.

# YOUR CODE HERE


In [ ]:
#@title Stretch Solution 3d
text = "The prime minister of the United Kingdom met with the president of France"
attn, tokens = get_attention(gpt2_model, gpt2_tokenizer, text)

seq_len = attn.shape[-1]
distances = torch.abs(
    torch.arange(seq_len).unsqueeze(0) - torch.arange(seq_len).unsqueeze(1)
).float()

avg_dists = []
for layer in range(attn.shape[0]):
    attn_avg = attn[layer].mean(dim=0)
    avg_dist = (attn_avg * distances).sum(dim=-1).mean().item()
    avg_dists.append(avg_dist)

plt.figure(figsize=(8, 4))
plt.bar(range(len(avg_dists)), avg_dists, color='steelblue')
plt.xlabel("Layer"); plt.ylabel("Average Attention Distance (tokens)")
plt.title("How Far Back Does Each Layer Look?")
plt.xticks(range(len(avg_dists)))
plt.tight_layout()
plt.show()

print("→ Early layers focus locally (nearby tokens).")
print("  Later layers develop longer-range attention patterns.")
print("  This is how the model builds from syntax to semantics.")

**Key takeaway from Section 4:** Attention lets each token "look at" every other token to decide what is relevant. You did this by hand in Exercise 3a: you assigned importance weights to words based on what would help predict the blank. The model does the same thing, but with learned Q, K, V transformations that run automatically. Multi-head attention lets it ask multiple questions simultaneously. And causal masking ensures it only looks backward.

---


---
# **Section 5: Language Modeling & Generation**
*Perplexity, decoding strategies, and the bridge to Day 2*

---

We have all the ingredients:
1. Words as meaningful vectors (embeddings)
2. Text as token sequences (tokenization)
3. A mechanism for tokens to share information (attention)
4. A model that uses all of this to predict the next token

Now let's look at how the model actually **generates text**, and how the choices we make during generation change the output dramatically.


## 5.1 Perplexity: Measuring Model Quality

**Perplexity** is the standard metric for language model quality. Intuitively, it answers: *"how surprised is the model by this text?"*

$$\text{Perplexity} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N}\log P(t_i \mid t_1, ..., t_{i-1})\right)$$

- **Lower perplexity = better model** (less "perplexed")
- Perplexity of 1 = perfect prediction
- Perplexity of 50,000 = random guessing from the vocabulary


### Exercise 4a: Computing Perplexity

Implement a function that computes perplexity, then compare it across different types of text.

**Steps:**
1. Tokenize the text and get model logits
2. For each position, get the log probability of the *actual* next token
3. Average the negative log probabilities
4. Exponentiate → perplexity


In [ ]:
def compute_perplexity(model, tokenizer, text):
    """
    Compute perplexity of text under the model.

    Steps:
    1. Tokenize and get model logits
    2. Convert logits to log probabilities: torch.log_softmax(logits, dim=-1)
    3. For position i (starting from 1), the target is input_ids[0, i]
       and the prediction is at log_probs[0, i-1, input_ids[0, i]]
    4. Average these log probs, then perplexity = exp(-average)
    """
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits  # (1, seq_len, vocab_size)

    # YOUR CODE HERE
    pass  # return a perplexity value


# Compare perplexity on different texts
texts = {
    "Normal English": "The government announced new economic policies to address growing concerns about inflation and unemployment across the country.",
    "Scrambled": "economic government announced growing the about new unemployment inflation address policies concerns and to the across country.",
    "Domain jargon": "The heteroscedasticity-consistent covariance matrix estimator under clustered standard errors with panel fixed effects.",
    "Repetitive": "the the the the the the the the the the the the the the the the the the the the",
}

for label, text in texts.items():
    ppl = compute_perplexity(gpt2_model, gpt2_tokenizer, text)
    print(f"  {label:25s} → perplexity: {ppl:>10.1f}")


In [ ]:
#@title Solution 4a
def compute_perplexity(model, tokenizer, text):
    """Compute perplexity of text under the model."""
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits

    log_probs = torch.log_softmax(logits, dim=-1)

    token_log_probs = []
    for i in range(1, input_ids.shape[1]):
        lp = log_probs[0, i - 1, input_ids[0, i]].item()
        token_log_probs.append(lp)

    avg_lp = sum(token_log_probs) / len(token_log_probs)
    return np.exp(-avg_lp)


texts = {
    "Normal English": "The government announced new economic policies to address growing concerns about inflation and unemployment across the country.",
    "Scrambled": "economic government announced growing the about new unemployment inflation address policies concerns and to the across country.",
    "Domain jargon": "The heteroscedasticity-consistent covariance matrix estimator under clustered standard errors with panel fixed effects.",
    "Repetitive": "the the the the the the the the the the the the the the the the the the the the",
}

print("Perplexity comparison (using GPT-2):\n")
for label, text in texts.items():
    ppl = compute_perplexity(gpt2_model, gpt2_tokenizer, text)
    print(f"  {label:25s} → perplexity: {ppl:>10.1f}")

print("\n→ Normal English: lowest perplexity (model expected this).")
print("  Scrambled: much higher (wrong word order is surprising).")
print("  Domain jargon: high (GPT-2 rarely saw this).")
print("  Repetitive: can be low ('the' after 'the' becomes predictable).")


## 5.2 Inside a Single Prediction Step

Before we generate text, let's understand what happens in a **single forward pass.** The model takes a sequence of tokens and outputs a score (called a **logit**) for every token in the vocabulary. These logits are then converted to probabilities.


In [ ]:
# One forward pass: prompt → logits → probabilities → predicted token
prompt = "The capital of France is"
inputs = gpt2_tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]

print(f"Prompt: '{prompt}'")
print(f"Token IDs: {input_ids[0].tolist()}")
print(f"Tokens: {[gpt2_tokenizer.decode([t]) for t in input_ids[0]]}\n")

# Forward pass
with torch.no_grad():
    outputs = gpt2_model(**inputs)

logits = outputs.logits          # shape: (1, seq_len, vocab_size)
next_logits = logits[0, -1, :]   # logits for the NEXT token: shape (vocab_size,)

print(f"Logits shape: {logits.shape}")
print(f"  → 1 batch × {logits.shape[1]} positions × {logits.shape[2]:,} vocab size")
print(f"  → Next-token logits: {next_logits.shape[0]:,} scores, one per vocabulary token\n")

# Convert logits to probabilities
probs = torch.softmax(next_logits, dim=-1)

# Top 10 predictions
top_probs, top_ids = probs.topk(10)
print("Top 10 predicted next tokens:")
for p, idx in zip(top_probs, top_ids):
    token = gpt2_tokenizer.decode([idx.item()])
    bar = "█" * int(p.item() * 80)
    print(f"  '{token}'  {p.item():.4f}  {bar}")

# Greedy choice: just take the argmax
greedy_id = next_logits.argmax()
print(f"\nGreedy pick (highest probability): '{gpt2_tokenizer.decode([greedy_id.item()])}'")

That is the complete picture of one step. To generate a full sequence, we repeat this in a loop: predict the next token, append it to the input, and predict again.


### Exercise 4b: Build a Greedy Text Generator

Write a function that generates text one token at a time using **greedy decoding** (always pick the highest-probability token).

**The algorithm:**
```
1. Tokenize the prompt → input_ids
2. Repeat max_new_tokens times:
   a. Forward pass: logits = model(input_ids).logits
   b. Get logits for the last position: next_logits = logits[0, -1, :]
   c. Pick the token with the highest logit: next_id = next_logits.argmax()
   d. Append next_id to input_ids
3. Decode input_ids back to text
```

**Hints:**
- `torch.cat([tensor_a, tensor_b], dim=1)` concatenates tensors along a dimension
- `next_id.unsqueeze(0).unsqueeze(0)` reshapes a scalar to shape (1, 1) for concatenation


In [ ]:
def generate_greedy(model, tokenizer, prompt, max_new_tokens=30):
    """Generate text using greedy decoding (always pick the most likely token)."""
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

    for _ in range(max_new_tokens):
        # YOUR CODE HERE
        # 1. Forward pass (inside torch.no_grad())
        # 2. Get next-token logits (last position)
        # 3. Pick the argmax
        # 4. Append to input_ids
        pass

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


# Test it
print(generate_greedy(gpt2_model, gpt2_tokenizer, "The capital of France is"))
print()
print(generate_greedy(gpt2_model, gpt2_tokenizer, "The future of democracy depends on"))

In [ ]:
#@title Solution 4b
def generate_greedy(model, tokenizer, prompt, max_new_tokens=30):
    """Generate text using greedy decoding."""
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids)
        next_logits = outputs.logits[0, -1, :]
        next_id = next_logits.argmax()
        input_ids = torch.cat([input_ids, next_id.unsqueeze(0).unsqueeze(0)], dim=1)

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


print(generate_greedy(gpt2_model, gpt2_tokenizer, "The capital of France is"))
print()
print(generate_greedy(gpt2_model, gpt2_tokenizer, "The future of democracy depends on"))
print()
print("→ Greedy decoding is deterministic: same prompt always gives same output.")
print("  Run it twice to verify. Notice it can get repetitive.")

### Exercise 4c: Add Temperature Scaling

Greedy decoding always picks the top token, which leads to repetitive, "safe" text. To introduce variety, we **sample** from the distribution instead of taking the argmax. Temperature controls how much variety:

$$\text{probs}_i = \frac{\exp(\text{logit}_i \;/\; T)}{\sum_j \exp(\text{logit}_j \;/\; T)}$$

This is just softmax with the logits divided by temperature $T$ before applying it.

- **T < 1.0**: distribution gets **sharper** (model becomes more confident, less random)
- **T = 1.0**: original distribution (unchanged)
- **T > 1.0**: distribution gets **flatter** (model considers more options, more random)

**Modify your generate function to:**
1. Divide logits by temperature before softmax
2. Convert to probabilities with `torch.softmax`
3. **Sample** from the distribution using `torch.multinomial(probs, num_samples=1)` instead of argmax

**Hint:** `torch.multinomial(probs, 1)` draws one sample from a probability distribution.


In [ ]:
def generate_with_temperature(model, tokenizer, prompt, max_new_tokens=30,
                             temperature=1.0):
    """Generate text with temperature-controlled sampling."""
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids)
        next_logits = outputs.logits[0, -1, :]

        # YOUR CODE HERE
        # 1. Divide logits by temperature
        # 2. Convert to probabilities with softmax
        # 3. Sample from the distribution with torch.multinomial
        # 4. Append sampled token to input_ids
        pass

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


# Compare temperatures
prompt = "The future of democracy depends on"
for temp in [0.3, 0.7, 1.0, 1.5]:
    text = generate_with_temperature(gpt2_model, gpt2_tokenizer, prompt, temperature=temp)
    print(f"  [T={temp}] {text}")
    print()

In [ ]:
#@title Solution 4c
def generate_with_temperature(model, tokenizer, prompt, max_new_tokens=30,
                             temperature=1.0):
    """Generate text with temperature-controlled sampling."""
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids)
        next_logits = outputs.logits[0, -1, :]

        # Temperature scaling
        scaled_logits = next_logits / temperature
        probs = torch.softmax(scaled_logits, dim=-1)

        # Sample from the distribution
        next_id = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_id.unsqueeze(0)], dim=1)

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


prompt = "The future of democracy depends on"
for temp in [0.3, 0.7, 1.0, 1.5]:
    text = generate_with_temperature(gpt2_model, gpt2_tokenizer, prompt, temperature=temp)
    print(f"  [T={temp}] {text}")
    print()

print("→ Low temperature (0.3): focused, repetitive, 'safe'.")
print("  High temperature (1.5): diverse but potentially incoherent.")
print("  Run this cell multiple times: unlike greedy, sampling gives different results each time.")

### Exercise 4d: Add Top-k and Top-p Filtering

Temperature alone can produce nonsense at high values because it samples from the *entire* vocabulary, including very unlikely tokens. Two filtering methods fix this by restricting which tokens are considered:

**Top-k filtering:** Keep only the k tokens with the highest probability. Set all others to zero, then re-normalize.

**Top-p (nucleus) filtering:** Keep the smallest set of tokens whose cumulative probability exceeds p. This adapts automatically: when the model is confident, fewer tokens pass the filter; when it is uncertain, more tokens pass.

**Modify your function to support both. The processing order is:**
```
logits → temperature scaling → top-k filter → top-p filter → softmax → sample
```

**Hints for top-k:**
- `torch.topk(logits, k)` returns the top-k values and their indices
- Set all logits below the k-th highest to `-float('inf')` so softmax gives them zero probability
- `values, indices = torch.topk(logits, k)` → `threshold = values[-1]`
- `logits[logits < threshold] = -float('inf')`

**Hints for top-p:**
- Sort probabilities in descending order: `sorted_probs, sorted_indices = torch.sort(probs, descending=True)`
- Compute cumulative sum: `cumulative = torch.cumsum(sorted_probs, dim=-1)`
- Find where cumulative probability exceeds p: `mask = cumulative - sorted_probs > top_p`
- Zero out everything past that point


In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=40,
             temperature=1.0, top_k=0, top_p=1.0):
    """
    Generate text with temperature, top-k, and top-p control.

    Args:
        temperature: controls randomness (lower = more focused)
        top_k: if > 0, only sample from the top-k most likely tokens
        top_p: if < 1.0, only sample from tokens with cumulative prob ≤ top_p
    """
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids)
        next_logits = outputs.logits[0, -1, :].clone()

        # Step 1: Temperature scaling
        next_logits = next_logits / temperature

        # Step 2: Top-k filtering
        if top_k > 0:
            # YOUR CODE HERE
            # Keep only the top_k highest logits, set the rest to -inf
            pass

        # Step 3: Top-p (nucleus) filtering
        if top_p < 1.0:
            # YOUR CODE HERE
            # Sort by probability, find the nucleus, mask out the tail
            pass

        # Step 4: Sample
        probs = torch.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_id.unsqueeze(0)], dim=1)

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


# Test all three controls
prompt = "The future of democracy depends on"

print("--- Greedy-ish (T=0.1) ---")
print(generate(gpt2_model, gpt2_tokenizer, prompt, temperature=0.1))
print("\n--- Temperature only (T=1.0) ---")
print(generate(gpt2_model, gpt2_tokenizer, prompt, temperature=1.0))
print("\n--- Top-k = 50 ---")
print(generate(gpt2_model, gpt2_tokenizer, prompt, temperature=1.0, top_k=50))
print("\n--- Top-p = 0.9 ---")
print(generate(gpt2_model, gpt2_tokenizer, prompt, temperature=1.0, top_p=0.9))
print("\n--- Top-k + Top-p + Temperature ---")
print(generate(gpt2_model, gpt2_tokenizer, prompt, temperature=0.8, top_k=50, top_p=0.9))

In [ ]:
#@title Solution 4d
def generate(model, tokenizer, prompt, max_new_tokens=40,
             temperature=1.0, top_k=0, top_p=1.0):
    """Generate text with temperature, top-k, and top-p control."""
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids)
        next_logits = outputs.logits[0, -1, :].clone()

        # Temperature scaling
        next_logits = next_logits / temperature

        # Top-k filtering
        if top_k > 0:
            top_values, _ = torch.topk(next_logits, top_k)
            threshold = top_values[-1]
            next_logits[next_logits < threshold] = -float('inf')

        # Top-p (nucleus) filtering
        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(next_logits, descending=True)
            sorted_probs = torch.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            # Remove tokens with cumulative probability above threshold
            # Shift right so the first token above p is kept
            sorted_mask = cumulative_probs - sorted_probs > top_p
            sorted_logits[sorted_mask] = -float('inf')

            # Unsort: put filtered logits back in original order
            next_logits = torch.zeros_like(next_logits).scatter_(0, sorted_indices, sorted_logits)

        # Sample
        probs = torch.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_id.unsqueeze(0)], dim=1)

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


prompt = "The future of democracy depends on"

print("--- Greedy-ish (T=0.1) ---")
print(generate(gpt2_model, gpt2_tokenizer, prompt, temperature=0.1))
print("\n--- Temperature only (T=1.0) ---")
print(generate(gpt2_model, gpt2_tokenizer, prompt, temperature=1.0))
print("\n--- Top-k = 50 ---")
print(generate(gpt2_model, gpt2_tokenizer, prompt, temperature=1.0, top_k=50))
print("\n--- Top-p = 0.9 ---")
print(generate(gpt2_model, gpt2_tokenizer, prompt, temperature=1.0, top_p=0.9))
print("\n--- Top-k + Top-p + Temperature ---")
print(generate(gpt2_model, gpt2_tokenizer, prompt, temperature=0.8, top_k=50, top_p=0.9))

print("\n→ Top-k and top-p prevent sampling from the long tail of unlikely tokens.")
print("  In practice, most LLM APIs use top-p=0.9 or similar as default.")
print("  The combination of temperature + top-p is the standard for production use.")

### Exercise 4e: Explore Generation on Social Science Prompts

Now use the `generate` function you built to explore how decoding parameters affect the "opinions" and style of model outputs on social science topics.

**Tasks:**
1. Try the prompt "The main cause of political polarization is" with low (0.3) and high (1.2) temperature. How do the completions differ in content and confidence?
2. Try "Immigration policy should" with different top-p values (0.5, 0.9, 1.0). How does the range of expressed positions change?
3. Come up with your own social science prompt. Can you find settings that produce more "conservative" vs. "progressive" sounding outputs?

**Think about:** If you were using an LLM to simulate survey responses (Day 4), how would these parameters affect the distribution of "opinions" you get?


In [ ]:
# YOUR EXPLORATION HERE

# Prompt ideas:
# "The main cause of political polarization is"
# "Immigration policy should"
# "The most important factor in economic growth is"
# "The role of government in healthcare"



---
## What We Covered Today

You now have a hands-on understanding of the **key building blocks** behind modern language models:

| Section | Key Idea | Why It Matters |
|---------|----------|----------------|
| **Representing Meaning** | Words as vectors with meaningful geometry | Embeddings encode cultural knowledge (and biases) |
| **From Static to Contextual** | Modern LMs understand words *in context* | The leap from GloVe to transformers |
| **Tokenization** | Subword units bridge text and computation | The tokenizer is part of your measurement instrument |
| **Attention** | Each token decides what is relevant | The mechanism that makes transformers scale |
| **Generation** | Next-token prediction + decoding strategies | Temperature and sampling shape model "behavior" |

## The Base Model → Assistant Gap

Everything we explored today was with **base models**: models trained only on next-token prediction. They are powerful pattern completers, but they are not assistants. They do not follow instructions, they do not refuse harmful requests, and they do not have conversations.

Try it yourself:


In [ ]:
# A base model does not follow instructions. It COMPLETES text.
prompt = "Please summarize the following article about climate change:"
print("Prompt:", prompt)
print("\nGPT-2 completion:")
print(generate(gpt2_model, gpt2_tokenizer, prompt, temperature=0.7, top_p=0.9))

print("\n→ The base model does not 'understand' this as an instruction.")
print("  It just continues the text pattern. Making models actually USEFUL")
print("  requires post-training: instruction tuning, RLHF, and more.")
print("  That is what Day 2 is about.")


## What is Next

Looking back at the course roadmap:

| Day | Theme | Status |
|-----|-------|--------|
| Day 1 | Foundations | ✓ Done |
| **→ Day 2** | **From Models to Tools** | **Up next** |
| Day 3 | Deploying for Research | |
| Day 4 | Social Science Applications | |
| Day 5 | Agentic Workflows | |

**Day 2** builds directly on today. Now that you understand *how* language models work, you will learn *how to use them effectively*:

- **Post-training**: how base models become useful assistants (RLHF, DPO, instruction tuning)
- **Prompting**: the core skill for working with LLMs, and why it is harder than it looks
- **Reasoning**: chain-of-thought, reasoning models, and when thinking helps
- **Evaluation**: how to choose the right model for your research task

Everything we covered today (embeddings, tokenization, attention, generation) directly informs how you design prompts, interpret model behavior, and build research pipelines in the days ahead.


---
*This notebook is part of the [LLMs for Social Science](https://llmsforsocialscience.net) course.*
